# Basketball feature availability and archive overlap

A feature needs an honest availability clock; an evaluation needs an honest comparison population. This notebook demonstrates the first on **invented teams, scores and clocks**, then inspects the second through **sanitized real saved-prediction aggregates**. It performs zero fits, model inference, source requests or new forecasts.

B006 is now authenticated as `SUCCESS_EVIDENCE_RECOVERY`. B003 remains the latest completed modeling release; B004 remains stopped; 238 real units and all 330 prior forecast versions remain preserved.

[Feature contract](../docs/BASKETBALL_FEATURE_RELEASE.md) · [Offline dashboard](../docs/basketball/index.html) · [Prior MARKET cohort study](../docs/BASKETBALL_MARKET_COVERAGE.md)

In [1]:
import json
from dataclasses import replace
from html import escape
from IPython.display import display, HTML
import plotly.graph_objects as go
import plotly.io as pio
from examples.basketball_contracts.feature_release import fixture, snapshot, IssuedSnapshots
from examples.basketball_contracts.build_basketball_dashboard import load_evidence, validate
pio.renderers.default="plotly_mimetype"
evidence=load_evidence()
print(json.dumps(validate(evidence),sort_keys=True))
records,request=fixture()
strict=snapshot(records,request)
research=snapshot(records,replace(request,timing="reconstructed_24h"))
assert strict["features"]["Amber"]["rolling_points_for"] == 104
assert research["features"]["Amber"]["prior_game_count"] == 3
print("Invented feature fixture only; real saved aggregate review remains separate.")

{"overlap_specifications": 120, "populated_panels": 24, "scorecards": 235}
Invented feature fixture only; real saved aggregate review remains separate.


## 1. Four clocks are not interchangeable

Each invented observation retains completion, publication, observation and revision identity. Strict eligibility requires an asserted original publication clock and both publication and observation before cutoff. **The verification flags here are synthetic caller assertions, not independently verified publication evidence.**

The correction to `invented-b` is released after cutoff, so the previous revision supplies the issued features. A backdated publication cannot defeat a later observation clock. The target game's own outcome is excluded by event identity even if a timestamp claims earlier availability.

In [2]:
fig=go.Figure()
for field,label,color,symbol in [("completed_at","Completed","#687990","circle"),("published_at","Claimed original publication","#1a72be","diamond"),("observed_at","Locally observed","#b26427","square")]:
    visible=[r for r in records if getattr(r,field) is not None]
    fig.add_trace(go.Scatter(x=[getattr(r,field) for r in visible],y=[f"{r.event} / r{r.revision}" for r in visible],mode="markers",name=label,marker={"size":10,"color":color,"symbol":symbol},customdata=[[r.team,r.original_clock_verified]for r in visible],hovertemplate="%{y}<br>%{x}<br>Invented team=%{customdata[0]}<br>Asserted clock flag=%{customdata[1]}<extra>%{fullData.name}</extra>"))
fig.add_shape(type="line",x0=request.cutoff,x1=request.cutoff,y0=0,y1=1,yref="paper",line={"dash":"dot","color":"#ad3a45"})
fig.add_annotation(x=request.cutoff,y=1.06,yref="paper",text="Pregame cutoff",showarrow=False,font={"color":"#ad3a45"})
fig.update_layout(title="Invented timeline: availability, correction and target-game exclusion",height=500,template="plotly_white",xaxis={"type":"date","title":"UTC time"},legend={"orientation":"h","y":-0.25},margin={"l":220,"r":30,"t":70,"b":125})
fig.show()

In [3]:
def table(headers,rows):
    head="".join("<th>"+escape(str(v))+"</th>"for v in headers)
    body="".join("<tr>"+"".join("<td>"+escape(str(v))+"</td>"for v in row)+"</tr>"for row in rows)
    return HTML("<table><thead><tr>"+head+"</tr></thead><tbody>"+body+"</tbody></table>")
display(table(["Protocol","Invented team","Eligible prior games","Rolling points for","Rolling points against","Missing history"],[[protocol,team,f["prior_game_count"],f["rolling_points_for"],f["rolling_points_against"],f["missing_history"]]for protocol,value in [("Strict asserted clocks",strict),("Reconstructed 24h research",research)]for team,f in value["features"].items()]))
issued=IssuedSnapshots()
before=snapshot(records[:2]+records[3:4],request)
assert issued.issue(before)=="issued"
assert strict["snapshot_sha256"]==before["snapshot_sha256"]
assert issued.issue(strict)=="verified_reuse"
print("Later excluded archive records leave the issued feature identity unchanged.")
print("Strict excluded counts:",strict["excluded_counts"])
print("The reconstructed protocol remains research; it does not authenticate source timing.")

Later excluded archive records leave the issued feature identity unchanged.
Strict excluded counts: {'other_scope': 0, 'target_or_future_outcome': 1, 'unverified_original_clock': 1, 'not_yet_available': 2}
The reconstructed protocol remains research; it does not authenticate source timing.


Protocol,Invented team,Eligible prior games,Rolling points for,Rolling points against,Missing history
Strict asserted clocks,Amber,2,104.0,100.0,False
Strict asserted clocks,Cobalt,1,95.0,97.0,False
Reconstructed 24h research,Amber,3,109.33333333333333,96.66666666666667,False
Reconstructed 24h research,Cobalt,1,95.0,97.0,False


## 2. A different overlap target, with support made visible

The new fixed-cell analysis uses empirical coverage fractions to construct overlap weights. It trains no propensity estimator and asserts no causal effect. All 120 declared specifications across six league/year groups are retained in the public evidence; the dashboard exposes them separately.

For NCAA men 2026, month × baseline-favorite-confidence × game-type overlap weighting at minimum one observation per arm retains **100% covered and 76.73% uncovered**. The improvement gap is +0.001926415, with seven-day interval [−0.001768084, +0.004972057] and fourteen-day interval [−0.001289230, +0.004549375]. Rich-model log loss remains +0.009396337 worse than MARKET on the weighted covered population.

This is a **different target and binning specification** from PR21's covered-reference probability-quintile analysis. Neither supersedes the other numerically. Increasing minimum support removes more games; an interval crossing zero does not establish model equivalence or representative early-season MARKET performance.

In [4]:
specs=[s for s in evidence["overlap"]["overlap_sensitivities"] if s["league"]=="ncaam" and s["year"]==2026 and s["specification"]=="month_confidence_type" and not s["trim_e_05_95"]]
fig=go.Figure()
for block,color,symbol in [("7","#1763b9","circle"),("14","#aa5d20","diamond-open")]:
    x=[s["overlap_gap"]for s in specs];intervals=[s["intervals"][block]["gap_ci95"]for s in specs]
    fig.add_trace(go.Scatter(x=x,y=[f"Minimum {s['minimum_per_arm']} per arm"for s in specs],mode="markers",name=block+"-day blocks",marker={"color":color,"size":11,"symbol":symbol},error_x={"type":"data","symmetric":False,"array":[ci[1]-v for ci,v in zip(intervals,x)],"arrayminus":[v-ci[0]for ci,v in zip(intervals,x)]},customdata=[[100*s["covered_retention"],100*s["uncovered_retention"],s["overlap_uncovered_ess"]]for s in specs],hovertemplate="%{y}<br>Gap=%{x:.9f}<br>Covered retained=%{customdata[0]:.2f}%<br>Uncovered retained=%{customdata[1]:.2f}%<br>Uncovered ESS=%{customdata[2]:.1f}<extra>%{fullData.name}</extra>"))
fig.add_vline(x=0,line_color="#555",line_dash="dot")
fig.update_layout(title="NCAA men 2026: overlap gap and block-length sensitivity",xaxis_title="Covered − uncovered rich-model improvement",height=390,template="plotly_white",margin={"l":185,"r":30,"t":65,"b":70},legend={"orientation":"h"})
fig.show()
display(table(["Minimum per arm","Covered retained","Uncovered retained","Covered ESS","Uncovered ESS","Gap","Selected minus MARKET LL"],[[s["minimum_per_arm"],f'{s["covered_retention"]:.2%}',f'{s["uncovered_retention"]:.2%}',f'{s["overlap_covered_ess"]:.1f}',f'{s["overlap_uncovered_ess"]:.1f}',f'{s["overlap_gap"]:+.9f}',f'{s["overlap_selected_minus_market_ll"]:+.9f}']for s in specs]))

Minimum per arm,Covered retained,Uncovered retained,Covered ESS,Uncovered ESS,Gap,Selected minus MARKET LL
1,100.00%,76.73%,1129.3,2443.1,+0.001926415,+0.009396337
5,98.69%,61.57%,1116.8,2359.0,+0.002588833,+0.010248203
20,93.61%,47.03%,1065.2,2099.7,+0.002367095,+0.011772052


In [5]:
display(table(["League","Season","Saved-model games","Covered","Teams with zero covered appearances","All saved-model teams"],[[r["league"],r["year"],r["saved_model_games"],r["covered_games"],next(t["teams_zero_covered_appearances"]for t in evidence["overlap"]["team_coverage_aggregate"]if t["league"]==r["league"]and t["year"]==r["year"]),next(t["teams_with_saved_games"]for t in evidence["overlap"]["team_coverage_aggregate"]if t["league"]==r["league"]and t["year"]==r["year"])]for r in evidence["overlap"]["league_year_coverage"]]))
print("Team appearances are not independent games. Men 2025 has no covered comparison.")
print("NBA and women's sparse coverage prevents importing the men's result as a general explanation.")

Team appearances are not independent games. Men 2025 has no covered comparison.
NBA and women's sparse coverage prevents importing the men's result as a general explanation.


League,Season,Saved-model games,Covered,Teams with zero covered appearances,All saved-model teams
nba,2025,1248,9,15,30
nba,2026,1249,16,12,30
ncaam,2025,5758,0,364,364
ncaam,2026,5749,1143,79,365
ncaaw,2025,5280,149,284,362
ncaaw,2026,5621,143,238,363


## Recovery is not a new modeling release

B006 authenticated its saved analysis, 128 return pins, 423 embedded-source pins and 222 inherited native references without loading models. It added no fits, inference, source requests, forecasts or orders. B003 remains the latest fully completed modeling release. All 330 forecast versions across 61 games remain preserved; there is no new settlement check.

The maximum measured floor is **4,263.755538637 seconds**. The open finalization reservation through **4,443.755538637 seconds is not measured spend**. Conservative headroom of **56.244461363 seconds** is below the unchanged 60-second guard until an exact matching closed receipt is authenticated. Public requests are 48/48. No additional research budget is active.

Source reconciliation has admitted no canonical player repair or pregame player-feature row. Dynamic-strength integration, synthetic checks and public feature contracts are not real trained results. Current 2027 outcomes remain monitoring-only and never choose features, hyperparameters, calibration, ensembles or champions. Portfolio assessment remains 8.5/10.

In [6]:
assert validate(evidence)["overlap_specifications"]==120
assert evidence["recovery"]["new_real_fits"]==0
assert evidence["recovery"]["cumulative_real_units"]==238
assert not evidence["overlap"]["current_2027_labels_used"]
assert not evidence["recovery"]["resources"]["additional_budget_authorized"]
print("FEATURE_RELEASE_REVIEW_COMPLETE")
print("Invented feature demonstration and real aggregate review completed separately; zero new model work or requests.")

FEATURE_RELEASE_REVIEW_COMPLETE
Invented feature demonstration and real aggregate review completed separately; zero new model work or requests.
